# Reward models and DPO

**Track:** Fine-tuning and alignment. **Level:** Advanced. **CPU:** yes.

## Goals

- Fit a reward model with a Bradley-Terry loss on synthetic preferences.
- Implement the DPO loss and check that the preferred response gains probability.
- Compare a low `beta` with a high `beta` by the KL from the reference policy.

## Sources and license

Original notebook. Bradley-Terry preference modeling is the reward-model loss in Christiano et al., *Deep Reinforcement Learning from Human Preferences* (2017) and in Ouyang et al. (InstructGPT, 2022). The direct preference loss is Rafailov et al., *Direct Preference Optimization* (2023):

`-log sigmoid( beta * (log pi(y_w)/pi_ref(y_w) - log pi(y_l)/pi_ref(y_l)) )`.

No human preference dataset is included. The pairs are drawn from a known linear reward so the accuracy has a right answer.

## Concepts

A reward model scores a response. Bradley-Terry says the preferred response should outscore the rejected one, with probability `sigmoid(r_w - r_l)`. DPO writes the same objective on the policy itself, using the closed form that relates the optimal policy to a reference policy and a reward. `beta` is how hard the policy is pulled toward the reference. At the optimum, a large `beta` stays closer to the reference. A small `beta` is willing to put nearly all mass on the preferred action. The KL measured at the end of training is the check.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'torch', 'numpy'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

import torch
import torch.nn as nn
import torch.nn.functional as F

from labs._shared.seed import seed_everything

seed_everything(0)
torch.manual_seed(0)


## Guided exercise: Bradley-Terry reward model

Two actions. The preferred action is the one with the higher linear score `x · w_a`. The reward model is a linear map from `x` to two scores. Accuracy is how often the higher score matches the preference. The features are random and the rule is linear, so this should not be a close call.


In [ ]:
N, D = 900, 6
g = torch.Generator().manual_seed(0)
X = torch.randn(N, D, generator=g)
W = torch.randn(2, D, generator=g)
true_score = X @ W.T
prefer_second = true_score[:, 1] > true_score[:, 0]

reward = nn.Linear(D, 2)
opt = torch.optim.Adam(reward.parameters(), lr=0.08)
for _ in range(80):
    scores = reward(X)
    chosen = torch.where(prefer_second, scores[:, 1], scores[:, 0])
    rejected = torch.where(prefer_second, scores[:, 0], scores[:, 1])
    loss = -F.logsigmoid(chosen - rejected).mean()
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
with torch.no_grad():
    accuracy = float((reward(X).argmax(dim=-1) == prefer_second.long()).float().mean())
print("reward loss", round(float(loss.detach()), 3), "accuracy", round(accuracy, 3))
assert accuracy > 0.85


## Guided exercise: DPO

The policy is a linear map to two logits. The reference is a frozen copy of the initialization. The loss is the expression above. The margin is the mean difference between the log-probability of the preferred action and the rejected action. It should rise.


In [ ]:
def dpo_loss(policy, reference, features, prefer_second, beta):
    logits = policy(features)
    with torch.no_grad():
        ref_logits = reference(features)
    logp = F.log_softmax(logits, dim=-1)
    logref = F.log_softmax(ref_logits, dim=-1)
    index_w = prefer_second.long()
    index_l = 1 - index_w
    pi_w = logp.gather(1, index_w[:, None]).squeeze(1)
    pi_l = logp.gather(1, index_l[:, None]).squeeze(1)
    ref_w = logref.gather(1, index_w[:, None]).squeeze(1)
    ref_l = logref.gather(1, index_l[:, None]).squeeze(1)
    return -F.logsigmoid(beta * ((pi_w - pi_l) - (ref_w - ref_l))).mean()

def margin(policy, features, prefer_second):
    logp = F.log_softmax(policy(features), dim=-1)
    index_w = prefer_second.long()
    index_l = 1 - index_w
    return float((logp.gather(1, index_w[:, None]) - logp.gather(1, index_l[:, None])).mean())

def kl_to_ref(policy, reference, features):
    logp = F.log_softmax(policy(features), dim=-1)
    logref = F.log_softmax(reference(features), dim=-1)
    probs = logp.exp()
    return float((probs * (logp - logref)).sum(dim=-1).mean())

def train_dpo(beta, steps=120, lr=0.08):
    torch.manual_seed(2)
    policy = nn.Linear(D, 2)
    reference = nn.Linear(D, 2)
    reference.load_state_dict(policy.state_dict())
    for p in reference.parameters():
        p.requires_grad_(False)
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    before = margin(policy, X, prefer_second)
    for _ in range(steps):
        loss = dpo_loss(policy, reference, X, prefer_second, beta)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    after = margin(policy, X, prefer_second)
    return before, after, kl_to_ref(policy, reference, X), float(loss.detach())

before, after, kl_low, _loss = train_dpo(0.2)
print("margin", round(before, 3), "->", round(after, 3), "KL at beta 0.2", round(kl_low, 3))
assert after > before + 0.2


## Your turn

Write down the loss at `beta = 1` when the policy and the reference assign the same probability to both answers. `logsigmoid(0)` is `-log(0.5)`. The loss should be that number, up to floating point.

## Solution


In [ ]:
torch.manual_seed(0)
policy = nn.Linear(D, 2)
reference = nn.Linear(D, 2)
reference.load_state_dict({k: v.detach().clone() for k, v in policy.state_dict().items()})
for p in reference.parameters():
    p.requires_grad_(False)
value = float(dpo_loss(policy, reference, X[:64], prefer_second[:64], beta=1.0))
print("loss at the reference", round(value, 4), "target", round(float(-torch.log(torch.tensor(0.5))), 4))
assert abs(value - float(-torch.log(torch.tensor(0.5)))) < 1e-4


## Beyond the course

Train again at `beta = 5`. The preferred-answer margin should still increase, and the KL to the reference should be smaller than it was at `beta = 0.2`. That is the knob the DPO paper uses to stay near the reference policy. It is measured here, not assumed.


In [ ]:
before_hi, after_hi, kl_high, _ = train_dpo(5.0)
print("beta 5 margin", round(before_hi, 3), "->", round(after_hi, 3), "KL", round(kl_high, 3))
print("beta 0.2 KL", round(kl_low, 3))
assert after_hi > before_hi
assert kl_high < kl_low


## Research questions

1. The preferences were noise-free. Where does Bradley-Terry accuracy stop being a measure of the reward model and start being a measure of label noise?
2. DPO never fits an explicit reward head. What question about the reward can you no longer ask after you throw the head away?
3. `beta` was constant. Gao et al. show reward over-optimization when the policy chases a learned reward too hard. Which quantity in this notebook would move first if the preferences were themselves the output of a fitted model?

## Further reading

- Christiano et al., *Deep Reinforcement Learning from Human Preferences* (2017).
- Ouyang et al., InstructGPT (OpenAI, 2022).
- Rafailov et al., *Direct Preference Optimization* (2023).
- Bai et al., *Training a Helpful and Harmless Assistant with Reinforcement Learning from Human Feedback* (Anthropic, 2022).
- Gao, Schulman, and Hilton, *Scaling Laws for Reward Model Overoptimization* (OpenAI, 2022).
